# Feature Attribution via Raking - v1.0

In [1]:
import sys
import time
import numpy as np
import pandas as pd
import zipfile as zf
import scipy.linalg as la
import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb

In [2]:
!{sys.executable} -m pip install catboost
from catboost import CatBoostClassifier, Pool

In [3]:
!kaggle competitions download -c titanic

titanic.zip: Skipping, found more recently modified local copy (use --force to force download)


In [4]:
ds= zf.ZipFile('titanic.zip')

train_data= pd.read_csv(ds.open('train.csv'))
test_data= pd.read_csv(ds.open('test.csv'))

train_data.shape, test_data.shape

((891, 12), (418, 11))

In [5]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [7]:
X_all= pd.concat([train_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']],
                   test_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]]).set_index('PassengerId')

y_train= train_data[['PassengerId','Survived']].set_index('PassengerId')['Survived']

X_all.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.2500,S
2,1,female,38.0,1,0,71.2833,C
3,3,female,26.0,0,0,7.9250,S
4,1,female,35.0,1,0,53.1000,S
5,3,male,35.0,0,0,8.0500,S


In [8]:
numeric_columns= ['Age','SibSp','Parch','Fare']
categor_columns= list(filter(lambda x:x not in numeric_columns,X_all.columns))

X_train= X_all.iloc[:len(train_data),:].copy()
X_test= X_all.iloc[len(train_data):].copy()

In [9]:
# categorical features: fill nan's with mode for each set, train and test
for col in categor_columns:
    ft_mode= X_train[col].value_counts().index[0]
    X_train[col]= X_train[col].fillna(ft_mode)
    
    ft_mode= X_test[col].value_counts().index[0]
    X_test[col]= X_test[col].fillna(ft_mode)

In [10]:
# numerical features: fill nan's with median for each set, train and test
for col in numeric_columns:
    ft_median= X_train[col].median()
    X_train[col]= X_train[col].fillna(ft_median)
    
    ft_median= X_test[col].median()
    X_test[col]= X_test[col].fillna(ft_median)

In [11]:
# one-hot encoding the qualitative features
X_train_ohe= pd.get_dummies(X_train,columns=categor_columns)

X_train_ohe.head()

,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
PassengerId,,,,,,,,,,,,
1,22.0,1,0,7.2500,0,0,1,0,1,0,0,1
2,38.0,1,0,71.2833,1,0,0,1,0,1,0,0
3,26.0,0,0,7.9250,0,0,1,1,0,0,0,1
4,35.0,1,0,53.1000,1,0,0,1,0,0,0,1
5,35.0,0,0,8.0500,0,0,1,0,1,0,0,1


In [12]:
def normalize(df, cols):
    result= df.copy()
    
    for col in cols:
        max_value= df[col].max()
        min_value= df[col].min()
        result[col]= (df[col]- min_value)/ (max_value - min_value)
        
    return result

In [13]:
y_train.head()

PassengerId
1    0
2    1
3    1
4    1
5    0
Name: Survived, dtype: int64

In [14]:
# normalize the numeric columns of dataframe with each value between 0 and 1
X_train= normalize(X_train, numeric_columns)
X_train_ohe= normalize(X_train_ohe, numeric_columns)

In [15]:
# ML model setup
train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)

#train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
#xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)

#train, test, labels_train, labels_test= train_test_split(X_train,y_train,train_size=0.80,random_state=1234)
#cat_fts= [train.columns.to_list().index(col) for col in categor_columns]
#ctb_model= CatBoostClassifier(cat_features=cat_fts,silent=True)

In [16]:
# Define replace values (mode to categorical and mean to numeric) to fill train cols
col_mode= train.mode(axis=0)
col_mean= train.mean(axis=0)

for col in numeric_columns:
    col_mode[col]= col_mean[col]

replace_values= col_mode

In [17]:
acc_matrix= []
n_fts= len(train.columns)
start= time.time()

repeat_train= 5

for i in range(n_fts):
    acc_row= []
    
    for j in range(n_fts):
        trainings= []
        
        if (i!= j):
            # replace the j-ft with its respective mode/mean
            train_copy= train.copy()
            train_copy.loc[:,train_copy.columns[j]]= replace_values.iloc[0,j]
            
            for k in range(repeat_train):

                rf.fit(train_copy, labels_train)
                acc= sklearn.metrics.accuracy_score(labels_test, rf.predict(test))
                
                #xgb_model.fit(train_copy, labels_train)
                #acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))
                
                #ctb_model.fit(train_copy, labels_train)
                #acc= sklearn.metrics.accuracy_score(labels_test, ctb_model.predict(test))

                trainings.append(acc)
        
            acc_row.append(np.mean(trainings))
        else:
            acc_row.append(0)
        
    acc_matrix.append(acc_row)
    
end= time.time()
print("--- %s seconds ---" % np.round((end- start), 2))

--- 600.84 seconds ---


In [26]:
print(acc_matrix)
print(np.around(acc_matrix, decimals=3))

X_train_ohe.columns

[[0, 0.8201117318435754, 0.8156424581005586, 0.8312849162011172, 0.8256983240223464, 0.8201117318435754, 0.8212290502793296, 0.8212290502793296, 0.8212290502793296, 0.8189944134078212, 0.8201117318435754, 0.8145251396648044], [0.805586592178771, 0, 0.8167597765363128, 0.8256983240223464, 0.8256983240223464, 0.8212290502793296, 0.8234636871508381, 0.8212290502793296, 0.8201117318435754, 0.8223463687150838, 0.8201117318435754, 0.8178770949720671], [0.8, 0.8156424581005586, 0, 0.8279329608938548, 0.8268156424581006, 0.8212290502793296, 0.8212290502793296, 0.8212290502793296, 0.8201117318435754, 0.8212290502793296, 0.8189944134078212, 0.8122905027932961], [0.8, 0.8167597765363128, 0.8234636871508381, 0, 0.8268156424581006, 0.8212290502793296, 0.8201117318435754, 0.8201117318435754, 0.8201117318435754, 0.8189944134078212, 0.8201117318435754, 0.8189944134078212], [0.8, 0.8189944134078212, 0.8178770949720671, 0.8290502793296091, 0, 0.8212290502793296, 0.8201117318435754, 0.8201117318435754, 0

Index(['Age', 'SibSp', 'Parch', 'Fare', 'Pclass_1', 'Pclass_2', 'Pclass_3',
       'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S'],
      dtype='object')

In [19]:
eigvals, eigvecs= la.eig(acc_matrix)
eigvecs.shape

(12, 12)

In [28]:
print(eigvals)

[ 9.01289923+0.j         -0.79960023+0.j         -0.81166275+0.j
 -0.82651935+0.00276405j -0.82651935-0.00276405j -0.82570851+0.j
 -0.81848173+0.00110321j -0.81848173-0.00110321j -0.82096391+0.00073079j
 -0.82096391-0.00073079j -0.82199888+0.00014766j -0.82199888-0.00014766j]


In [29]:
print(eigvecs)

[[ 0.28918174+0.j          0.89414348+0.j          0.01920309+0.j
   0.08571893-0.04090147j  0.08571893+0.04090147j -0.02432874+0.j
   0.05475028-0.01937213j  0.05475028+0.01937213j -0.03813652+0.00902541j
  -0.03813652-0.00902541j -0.0221565 -0.0234719j  -0.0221565 +0.0234719j ]
 [ 0.28888589+0.j          0.19719524+0.j         -0.16710415+0.j
  -0.34403785+0.07218743j -0.34403785-0.07218743j  0.06920374+0.j
  -0.58688081+0.j         -0.58688081-0.j         -0.3225476 -0.05228136j
  -0.3225476 +0.05228136j  0.15483955-0.18751936j  0.15483955+0.18751936j]
 [ 0.28849177+0.j         -0.01760072+0.j          0.34161818+0.j
  -0.15644942-0.00211122j -0.15644942+0.00211122j -0.15261179+0.j
   0.18639671-0.04911918j  0.18639671+0.04911918j -0.09573687-0.00280241j
  -0.09573687+0.00280241j -0.06591205-0.05115098j -0.06591205+0.05115098j]
 [ 0.28849184+0.j         -0.03678429+0.j         -0.03751416+0.j
  -0.37383756+0.34599795j -0.37383756-0.34599795j -0.08062143+0.j
   0.32400777-0.04721899j

In [20]:
f= open('FAR_data.txt', 'w')

for i in range(n_fts):
    for j in range(n_fts):
        line= (str(i) + ',' + str(j) + ',' + str(acc_matrix[i][j]) + '\n')
        f.write(line)
        
f.close()

In [21]:
import sklearn.datasets

iris= sklearn.datasets.load_iris()
train, test, labels_train, labels_test= train_test_split(iris.data,iris.target,train_size=0.80,random_state=1234)

In [22]:
rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2,random_state=0)
rf.fit(train, labels_train)
sklearn.metrics.accuracy_score(labels_test, rf.predict(test))

1.0

In [23]:
xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train)
sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

1.0

In [24]:
ctb_model= CatBoostClassifier(silent=True)
ctb_model.fit(train,labels_train)
sklearn.metrics.accuracy_score(labels_test,ctb_model.predict(test))

1.0